In [ ]:
# Useful libraries
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install numpy
!pip install fsspec
!pip install s3fs
!pip install requests
import os
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
from sklearn import preprocessing
from sklearn.mixture import GaussianMixture


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [11]:
df_portefeuille = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_portefeuille_2023_11.csv", sep=";")
df_finalmmations = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_consommations_2023_11.csv", sep=";")
df_reclamations = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_reclamations_2023_11.csv", sep=";")
df_interactions = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_interactions_2023_11.csv", sep=";")
df_impayes = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_impayes_2023_11.csv", sep=";")

/tmp/ipykernel_613226/1075442604.py:1: DtypeWarning: Columns (31,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df_portefeuille = pd.read_csv("s3://projet-bdc-data/alptis/training/apprentissage_portefeuille_2023_11.csv", sep=";")


In [12]:
# Import the aggregation module
import sys
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)

from utils import TransformData, ModelData
from merge_and_agregation import aggregate_impayes, aggregate_interaction, aggregate_consommations, aggregate_reclamations

In [13]:
# Aggretation
df1 = aggregate_consommations(df_finalmmations)
df2 = aggregate_reclamations(df_reclamations)
df3 = aggregate_interaction(df_interactions)
df4 = aggregate_impayes(df_impayes)

/home/onyxia/work/bdc-alptis-g2/merge_and_agregation.py:252: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("client_code").apply(build_info_dict).reset_index(drop=True)
/home/onyxia/work/bdc-alptis-g2/merge_and_agregation.py:377: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_interaction_historique_mail = df_mail.groupby("client_code").apply(


In [14]:
# Final dataset
df_final = (
    df_portefeuille
    .set_index("client_code")
    .join(df1.set_index("client_code"), how="left")
    .join(df2.set_index("client_code"), how="left")
    .join(df3.set_index("client_code"), how="left")
    .join(df4.set_index("client_code"), how="left")
    .reset_index()
)

In [15]:
# Format the variables
df_final['client_code'] = df_final['client_code'].astype(str)
df_final['client_code_postal'] = df_final['client_code_postal'].astype(str).str.zfill(5)
df_final['client_departement'] = df_final['client_departement'].astype(str).str.zfill(2)
df_final['client_departement_avant_changement_12_mois'] = df_final['client_departement_avant_changement_12_mois'].astype(str)
df_final['client_a_garantie_prevoyance'] = df_final['client_a_garantie_prevoyance'].astype(str)
df_final['client_est_contrat_madelin'] = df_final['client_est_contrat_madelin'].astype(str)
df_final['courtier_est_escompte'] = df_final['courtier_est_escompte'].astype(str)
df_final['client_a_conjoint'] = df_final['client_a_conjoint'].astype(str)
df_final['courtier_code_partenaire'] = df_final['courtier_code_partenaire'].astype(str)
df_final['courtier_code_apporteur'] = df_final['courtier_code_apporteur'].astype(str)
df_final['courtier_code_avant_changement_12_mois'] = df_final['courtier_code_avant_changement_12_mois'].astype(str)
df_final['client_nb_enfants'] = df_final['client_nb_enfants'].astype(int)
df_final['client_nb_assures_contrat'] = df_final['client_nb_assures_contrat'].astype(int)

## int
#df_final['nb_decomptes_hospitalisation'] = df_final['nb_decomptes_hospitalisation'].astype(int)
df_final['nb_decomptes_soins_medicaux'] = df_final['nb_decomptes_soins_medicaux'].fillna(0).astype(int)
df_final['nb_decomptes_pharmacie'] = df_final['nb_decomptes_pharmacie'].fillna(0).astype(int)
df_final['nb_decomptes_optique'] = df_final['nb_decomptes_optique'].fillna(0).astype(int)
df_final['nb_decomptes_dentaire'] = df_final['nb_decomptes_dentaire'].fillna(0).astype(int)
df_final['nb_jours_forfait_journalier'] = df_final['nb_jours_forfait_journalier'].fillna(0).astype(int)
df_final['nb_jours_chambre_particuliere'] = df_final['nb_jours_chambre_particuliere'].fillna(0).astype(int)
df_final['nb_protheses_dentaires'] = df_final['nb_protheses_dentaires'].fillna(0).astype(int)
df_final['nb_decomptes'] = df_final['nb_decomptes'].fillna(0).astype(int)
df_final['nb_decomptes_divers'] = df_final['nb_decomptes_divers'].fillna(0).astype(int)

## Columns from secondary dataset
for elem in list(df_final.select_dtypes(include=['float', 'int']).columns):
    df_final[elem] = df_final[elem].fillna(0).astype(float)

# running data type analysis
TransformData.grouptype(df_final)

{'object': ['client_code',
  'client_sexe_souscripteur',
  'client_ro_souscripteur',
  'client_code_postal',
  'client_departement',
  'client_departement_avant_changement_12_mois',
  'client_a_conjoint',
  'client_structure_familiale',
  'client_a_garantie_prevoyance',
  'client_produit',
  'client_garantie_base',
  'client_garantie_base_plus_options',
  'client_millesime_tarifaire_produit',
  'client_zone_tarifaire',
  'client_type_zone_tarifaire',
  'client_est_contrat_madelin',
  'client_date_debut_effet_garantie',
  'client_mode_paiement',
  'client_frequence_paiement',
  'client_nom_banque',
  'client_nps_date_reponse_n',
  'client_nps_verbatim_n',
  'client_nps_date_reponse_n_moins1',
  'client_nps_verbatim_n_moins1',
  'courtier_code_partenaire',
  'courtier_code_apporteur',
  'courtier_code_avant_changement_12_mois',
  'courtier_reseau_courtage',
  'courtier_segmentation_interne',
  'courtier_type_commission',
  'courtier_est_escompte',
  'annee_mois_paiement',
  'info_par_ann

In [16]:
# preprocessing before regressions
df_regress = df_final[
    [x for x in list(df_final.columns) if x.startswith('client') | x.startswith('nb_decomptes_') | x.startswith('remb_alptis_') | x.startswith('reste_a_charge_') 
    | x.startswith('target_resiliation_6mois')]
    ]
df_regress
df_regress.shape

(132576, 53)

In [17]:
# numerical variables
list_numerical = df_regress.select_dtypes(exclude=["object","bool"]).columns.to_list()

# Exclusion
exclusion = ['client_nb_assures_contrat', 'client_nps_note_reco_n', 'client_nps_note_reco_n_moins1', 'target_resiliation_6mois']
for elem in exclusion:
    list_numerical.remove(elem)
    
list_numerical

['client_age_souscripteur',
 'client_nb_enfants',
 'client_anciennete_contrat_annees',
 'client_cotisations_annualisees_n_moins2',
 'client_cotisations_annualisees_n_moins1',
 'client_cotisations_annualisees_n',
 'client_cotisations_annualisees_n_plus1',
 'nb_decomptes_hospitalisation',
 'nb_decomptes_soins_medicaux',
 'nb_decomptes_pharmacie',
 'nb_decomptes_optique',
 'nb_decomptes_dentaire',
 'nb_decomptes_divers',
 'remb_alptis_hospitalisation',
 'remb_alptis_soins_medicaux',
 'remb_alptis_pharmacie',
 'remb_alptis_optique',
 'remb_alptis_dentaire',
 'remb_alptis_divers',
 'reste_a_charge_hospitalisation',
 'reste_a_charge_soins_medicaux',
 'reste_a_charge_pharmacie',
 'reste_a_charge_optique',
 'reste_a_charge_dentaire',
 'reste_a_charge_divers']

In [ ]:
# GMM

X = df_regress[list_numerical]
varying_k = list(range(2,10))
dict_gmm = {}

for elem in X:
    
    gmm = GaussianMixture(n_components=elem, max_iter=250)
    gmm.fit(X=X)